In [1]:
# Now time to try masked training
import json
import time
import warnings
from collections import Counter

import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    AutoModelForMaskedLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers.utils import logging as hf_logging
model_id = "thomas-sounack/BioClinical-ModernBERT-base"


def load_pubmed_split(path):
    # Load one PubMedQA split into a Hugging Face Dataset. 
    with open(path, "r") as f:
        raw = json.load(f)

    rows = []
    for example_id, row in raw.items():
        rows.append(
            {
                "ID": example_id,
                "QUESTION": row["QUESTION"],
                "CONTEXTS": " ".join(row["CONTEXTS"])
            }
        )

    return Dataset.from_list(rows)

def tokenize_batch(batch):
    """
    Tokenise question + context pairs.

    We keep the question intact and only truncate the context if needed: truncation="only_second", second is context!.
    This is safer because the question is short and most critical.
    """
    return tokenizer(
        batch["QUESTION"],
        batch["CONTEXTS"],
        truncation="only_second",
        max_length=512,
        return_special_tokens_mask= True
    )


def mlm_training_args(num_train_epochs = 1, learning_rate = 2e-5, weight_decay = 0.0):
    training_args = TrainingArguments(
        output_dir="./mlm_runs",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps = 2,
        weight_decay= weight_decay,
        push_to_hub=False,
        save_total_limit=1,
        report_to="none",
        save_only_model=True,
        seed=7,
        data_seed=7,
        logging_strategy="epoch"
        
    )

    return training_args

def prop_dataset(unlabelled_masked_train, unlabelled_masked_test, mode = 'pilot'):
    if mode == 'pilot':      
        n_train = int(0.12*len(unlabelled_masked_train))
        n_test = int(0.1*len(unlabelled_masked_test))

    elif mode == 'full':
        n_train = int(len(unlabelled_masked_train))
        n_test = int(len(unlabelled_masked_test))

    unlabelled_masked_train = unlabelled_masked_train.select(range(n_train))
    unlabelled_masked_test = unlabelled_masked_test.select(range(n_test))

    return unlabelled_masked_train, unlabelled_masked_test

def tokenize_dataset_prop(unlabelled_masked_train, unlabelled_masked_test):
    tokenise_num_proc = 1
    
    unlabelled_masked_train = unlabelled_masked_train.map(
        tokenize_batch,
        batched = True,
        num_proc= tokenise_num_proc
    )
    
    unlabelled_masked_test = unlabelled_masked_test.map(
        tokenize_batch,
        batched = True,
        num_proc= tokenise_num_proc
    )
    
    # Now remove not needed columns
    unlabelled_masked_train = unlabelled_masked_train.remove_columns(
        ['ID', 'QUESTION', 'CONTEXTS']
    )
    
    unlabelled_masked_test = unlabelled_masked_test.remove_columns(
        ['ID', 'QUESTION', 'CONTEXTS']
    )

    return unlabelled_masked_train, unlabelled_masked_test
    
model = AutoModelForMaskedLM.from_pretrained(model_id)

unlabelled_path = '../Preprocessing+baseline/data/ori_pqau.json'

unlabelled = load_pubmed_split(unlabelled_path)

splitting = unlabelled.train_test_split(test_size = 0.2, seed = 7)

unlabelled_masked = splitting['train']
unlabelled_later_training = splitting['test']

splitting2 = unlabelled_masked.train_test_split(test_size = 1/8, seed = 7)

unlabelled_masked_train = splitting2['train']
unlabelled_masked_test = splitting2['test']

tokenizer = AutoTokenizer.from_pretrained(model_id)

data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm_probability= 0.15)


# First we do a pilot train to see if masked language training actually improves performance
unlabelled_masked_train_pilot, unlabelled_masked_test_pilot = prop_dataset(unlabelled_masked_train, unlabelled_masked_test, mode = 'pilot')
unlabelled_masked_train_pilot, unlabelled_masked_test_pilot = tokenize_dataset_prop(unlabelled_masked_train_pilot, unlabelled_masked_test_pilot)

trainer_pilot = Trainer(
    model = model,
    args = mlm_training_args(num_train_epochs = 8),
    train_dataset = unlabelled_masked_train_pilot,
    eval_dataset= unlabelled_masked_test_pilot,
    data_collator = data_collator,
    processing_class = tokenizer,
    callbacks= [EarlyStoppingCallback(early_stopping_patience = 2)]
)

train_output_pilot = trainer_pilot.train()

pilot_mlm_checkpoint = trainer_pilot.state.best_model_checkpoint
print("Best pilot MLM checkpoint:", pilot_mlm_checkpoint)

Map (num_proc=1):   0%|          | 0/5144 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/612 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,0.746200,0.762000
2,0.732600,0.743915
3,0.720300,0.759822
4,0.710800,0.775241


There were missing keys in the checkpoint model loaded: ['decoder.weight'].


Best pilot MLM checkpoint: ./mlm_runs/checkpoint-644


In [2]:
#Now to assess classification model

# keep output clean
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*Some weights of .* were not initialized.*")
hf_logging.set_verbosity_error()



# fixed label mapping
label_to_id = {"no": 0, "maybe": 1, "yes": 2}
id_to_label = {0: "no", 1: "maybe", 2: "yes"}

train_path = "../Preprocessing+baseline/data/pqal_fold0/train_set.json"
dev_path = "../Preprocessing+baseline/data/pqal_fold0/dev_set.json"


def print_divider(char="=", width=88):
    print(char * width)


def load_pubmedqa_split(path):
    # Load one PubMedQA split into a Hugging Face Dataset. this version has final_decision
    with open(path, "r") as f:
        raw = json.load(f)

    rows = []
    for example_id, row in raw.items():
        rows.append(
            {
                "ID": example_id,
                "QUESTION": row["QUESTION"],
                "CONTEXTS": " ".join(row["CONTEXTS"]),
                "final_decision": row["final_decision"].strip().lower(),
            }
        )

    return Dataset.from_list(rows)


def add_label(example):
    # Convert string labels to integer ids.
    example["labels"] = label_to_id[example["final_decision"]]
    return example


def build_fold0_dataset(tokenizer):
    """
    Build tokenised fold0 train/dev dataset for the classifier.
    This uses the exact same question + context setup as the base model (Repeating code because 
    importing from the base file reruns previous setup)
    """
    fold0_train = load_pubmedqa_split(train_path)
    fold0_dev = load_pubmedqa_split(dev_path)

    dataset = DatasetDict(
        {
            "train": fold0_train,
            "validation": fold0_dev,
        }
    ).map(add_label)

    dataset = dataset.map(tokenize_batch, batched=True)
    dataset = dataset.remove_columns(["ID", "QUESTION", "CONTEXTS", "final_decision"])

    return dataset


def compute_metrics(eval_pred):
    # Metrics used for fold0 comparison.
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    per_class_f1 = f1_score(
        labels,
        preds,
        average=None,
        labels=[0, 1, 2],
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_no": per_class_f1[0],
        "f1_maybe": per_class_f1[1],
        "f1_yes": per_class_f1[2],
    }


class SamplerOnlyTrainer(Trainer):
    """
    Trainer with weighted sampling only.
    This matches the best setup from the base ModernBERT experiments.
    """

    def __init__(self, *args, sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights = sample_weights

    def get_train_dataloader(self):
        sampler = WeightedRandomSampler(
            weights=self.sample_weights,
            num_samples=len(self.sample_weights),
            replacement=True,
        )

        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=False,
        )


def print_confusion_matrix_nicely(cm):
    # Print confusion matrix.
    print("\nConfusion matrix (rows = true, columns = predicted)")
    print(f"{'':<10}{'no':>8}{'maybe':>10}{'yes':>8}")
    labels = ["no", "maybe", "yes"]
    for label_name, row in zip(labels, cm):
        print(f"{label_name:<10}{row[0]:>8}{row[1]:>10}{row[2]:>8}")


def run_fold0_classifier_from_mlm(
    mlm_checkpoint_path,
    run_name,
    seed=42,
    learning_rate=8e-6,
    train_batch_size=8,
    eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    early_stopping_patience=2,
    phase_name="PILOT PHASE",
):
    """
    Load an MLM-adapted checkpoint into a sequence classifier, then
    fine-tune and evaluate on fold0 using the best base-model setup.

    This is meant to answer one question: does masked training improve downstream classification?
    """
    tokenizer = AutoTokenizer.from_pretrained(mlm_checkpoint_path)
    dataset = build_fold0_dataset(tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # sample weights for weighted sampler only
    train_labels = dataset["train"]["labels"]
    class_counts = Counter(train_labels)

    # inverse-frequency weights
    class_weights = {
        label_id: len(train_labels) / (3 * class_counts[label_id])
        for label_id in [0, 1, 2]
    }
    sample_weights = torch.tensor(
        [class_weights[label] for label in train_labels],
        dtype=torch.double,
    )

    def make_model():
        return AutoModelForSequenceClassification.from_pretrained(
            mlm_checkpoint_path,
            num_labels=3,
            id2label=id_to_label,
            label2id=label_to_id,
            ignore_mismatched_sizes=True,
        )

    training_args = TrainingArguments(
        output_dir=f"./classifier_runs/{run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        weight_decay=0.0,
        save_total_limit=1,
        save_only_model=True,
        report_to="none",
        disable_tqdm=True,
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=2,
    )

    start = time.perf_counter()

    trainer = SamplerOnlyTrainer(
        model_init=make_model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        sample_weights=sample_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    )

    trainer.train()
    eval_metrics = trainer.evaluate()

    pred_output = trainer.predict(dataset["validation"])
    preds = np.argmax(pred_output.predictions, axis=-1)
    labels = pred_output.label_ids
    cm = confusion_matrix(labels, preds, labels=[0, 1, 2])

    runtime_minutes = (time.perf_counter() - start) / 60.0

    result = {
        "run_name": run_name,
        "seed": seed,
        "mlm_checkpoint_path": mlm_checkpoint_path,
        "accuracy": eval_metrics["eval_accuracy"],
        "macro_f1": eval_metrics["eval_macro_f1"],
        "f1_no": eval_metrics["eval_f1_no"],
        "f1_maybe": eval_metrics["eval_f1_maybe"],
        "f1_yes": eval_metrics["eval_f1_yes"],
        "runtime_minutes": runtime_minutes,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
    }

    # clean printout
    print()
    print_divider()
    print(phase_name)
    print_divider()
    print("This is a quick downstream check of classification performance on fold0")
    print("using the MLM-adapted checkpoint plus the best base-model classifier setup.")
    print_divider()
    print(f"Run name           : {result['run_name']}")
    print(f"MLM checkpoint      : {result['mlm_checkpoint_path']}")
    print(f"Seed               : {result['seed']}")
    print(f"Learning rate      : {learning_rate}")
    print(f"Train batch size   : {train_batch_size}")
    print(f"Eval batch size    : {eval_batch_size}")
    print(f"Grad accumulation  : {gradient_accumulation_steps}")
    print(f"Runtime            : {result['runtime_minutes']:.2f} min")
    print()
    print("Fold0 dev results")
    print(f"  Accuracy         : {result['accuracy']:.4f}")
    print(f"  Macro F1         : {result['macro_f1']:.4f}")
    print(f"  F1 no            : {result['f1_no']:.4f}")
    print(f"  F1 maybe         : {result['f1_maybe']:.4f}")
    print(f"  F1 yes           : {result['f1_yes']:.4f}")
    print_confusion_matrix_nicely(cm)
    print_divider()

    return result, trainer

In [3]:
# run pilot MLM checkpoint through fold0 classifier

pilot_result, pilot_trainer = run_fold0_classifier_from_mlm(
    mlm_checkpoint_path=pilot_mlm_checkpoint,
    run_name="pilot_mlm_to_classifier_fold0",
    seed=42,
    learning_rate=8e-6,
    train_batch_size=8,
    eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    early_stopping_patience=2,
    phase_name="PILOT PHASE",
)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/var/folders/0j/h82t10zd3sq0dgxp1jrcmhzr0000gn/T/ipykernel_20754/3967849999.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SamplerOnlyTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


{'loss': 2.1367, 'grad_norm': 16.452014923095703, 'learning_rate': 7.034482758620689e-06, 'epoch': 1.0}
{'eval_loss': 1.1676751375198364, 'eval_accuracy': 0.34, 'eval_macro_f1': 0.30702551979147724, 'eval_f1_no': 0.2222222222222222, 'eval_f1_maybe': 0.23076923076923078, 'eval_f1_yes': 0.46808510638297873, 'eval_runtime': 21.0479, 'eval_samples_per_second': 2.376, 'eval_steps_per_second': 0.333, 'epoch': 1.0}
{'loss': 1.9496, 'grad_norm': 27.10396957397461, 'learning_rate': 6.0344827586206896e-06, 'epoch': 2.0}
{'eval_loss': 1.1104099750518799, 'eval_accuracy': 0.38, 'eval_macro_f1': 0.3032756716967243, 'eval_f1_no': 0.15384615384615385, 'eval_f1_maybe': 0.21052631578947367, 'eval_f1_yes': 0.5454545454545454, 'eval_runtime': 23.3302, 'eval_samples_per_second': 2.143, 'eval_steps_per_second': 0.3, 'epoch': 2.0}
{'loss': 1.8019, 'grad_norm': 15.606046676635742, 'learning_rate': 5.034482758620689e-06, 'epoch': 3.0}
{'eval_loss': 1.0859534740447998, 'eval_accuracy': 0.44, 'eval_macro_f1': 0